In [35]:
import os 
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns 

%store -r df_final df_prices_window

#### Data Quality / Validation

In [ ]:
df_eda = df_final.with_columns(
    pl.from_epoch("window_start_ts").dt.date().alias("market_date"),
    pl.when(pl.col("resolution").is_in([0, 1]))
    .then(pl.lit("known"))
    .otherwise(pl.lit("unknown"))
    .alias("resolution_status"),
)

# Table grain and timing checks
duplicate_markets = df_eda.height - df_eda["market_id"].n_unique()
duplicate_price_times = (
    df_prices_window
    .group_by(["market_id", "timestamp"])
    .len()
    .filter(pl.col("len") > 1)
    .height
)
future_prices = df_eda.filter(
    pl.col("price_timestamp") > pl.col("prediction_ts")
).height
invalid_prices = df_prices_window.filter(
    ~pl.col("up_price").is_between(0, 1, closed="both")
    | ~pl.col("down_price").is_between(0, 1, closed="both")
).height

# Missing five-minute market intervals
market_times = df_eda["window_start_ts"].unique().sort()
market_gaps = market_times.diff().drop_nulls()
gap_count = (market_gaps > 300).sum()
estimated_missing_intervals = (
    ((market_gaps.filter(market_gaps > 300) // 300) - 1).sum()
)

print(f"Rows / unique markets: {df_eda.height:,} / {df_eda['market_id'].n_unique():,}")
print(f"Duplicate market rows: {duplicate_markets:,}")
print(f"Duplicate (market, price timestamp) groups: {duplicate_price_times:,}")
print(f"Prices outside [0, 1]: {invalid_prices:,}")
print(f"Prices after prediction_ts: {future_prices:,}")
print(f"Gaps longer than five minutes: {gap_count:,}")
print(f"Estimated missing five-minute intervals: {estimated_missing_intervals:,}")

price_observations_per_market = (
    df_prices_window.group_by("market_id").len()["len"]
    .rename("price_observations_per_market")
)
up_plus_down = (
    df_eda["up_price"] + df_eda["down_price"]
).rename("up_plus_down")

display(
    df_eda.group_by("resolution").len().sort("resolution"),
    df_eda["price_age_seconds"].describe(),
    up_plus_down.describe(),
    price_observations_per_market.describe(),
)

# Coverage tables (ready to plot later if you add a plotting library)
daily_counts = (
    df_eda.group_by(["market_date", "resolution_status"])
    .len()
    .sort("market_date")
)
price_age_buckets = (
    df_eda
    .with_columns(
        pl.col("price_age_seconds").cut(
            [10, 20, 30, 45, 60, 120],
            labels=["0-10", "11-20", "21-30", "31-45", "46-60", "61-120", "120+"],
        ).alias("price_age_bucket")
    )
    .group_by("price_age_bucket")
    .len()
    .sort("price_age_bucket")
)

#display(daily_counts, price_age_buckets)

In [ ]:
# plots for daily counts / price age buckets 

output_dir = ("../reports/figures") 

sns.set_theme(style="whitegrid", palette="muted")

plt.figure(figsize=(12, 6))

sns.barplot(
    data=daily_counts,         
    x="market_date",
    y="len",
    hue="resolution_status"
)

plt.title("Daily Counts by Resolution Status", fontsize=14, pad=15)
plt.xlabel("Market Date", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()               

daily_counts_path = os.path.join(output_dir, "daily_counts.png")
plt.savefig(daily_counts_path, dpi=300)
plt.close()                    


plt.figure(figsize=(10, 5))

sns.barplot(
    data=price_age_buckets,     
    x="price_age_bucket",
    y="len",
    palette="viridis"       
)

plt.title("Volume Distribution by Price Age Buckets", fontsize=14, pad=15)
plt.xlabel("Price Age Bucket", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.tight_layout()


price_age_buckets_path = os.path.join(output_dir, "price_age_buckets.png")
plt.savefig(price_age_buckets_path, dpi=300)
plt.close()


C:\Users\Anigma PC\AppData\Local\Temp\ipykernel_4328\716370937.py:29: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(
